# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
import os
os.environ["JAVA_HOME"] = "/home/trar3243/miniforge3/envs/spark_env"

from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

from pathlib import Path

checkpoint_dir = Path("./spark_checkpoint").resolve()
checkpoint_dir.mkdir(parents=True, exist_ok=True)

sc.setCheckpointDir(checkpoint_dir.as_uri())

print(sc.getCheckpointDir())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 19:45:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


file:/home/trar3243/repos/datacenter_scale/lab4-pyspark-patent/spark_checkpoint/8f35cae6-3733-4698-99be-f86f7731c49e


Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [18]:
"""# goal: get patent alongside count, we will join back to patent later 
counts = patents.join(
        citations, on=patents.PATENT == citations.CITING, how="inner" # join to citations as "CITING" to figure out which patent his patent cites 
    ).alias(
        "first"
    ).join(
        patents.alias("second"), on= (col("first.cited") == col("second.patent")) & (col("first.postate") == col("second.postate")) # join back to patents to get all the patent information for what patent each orig patent cites 
    ).filter(
        (col("first.postate").isNotNull()) & (col("first.postate") != "") # filter out null and empty states 
    ).select(
        col("first.patent").alias("patent_to_drop") # select only the "citing" patent 
    ).groupBy(
        col("patent_to_drop") # group by the patent because each citing patent now has multiple record for each in-state cited patent 
    ).agg(
        F.count("*").alias("SAME_STATE") # get count 
    ).checkpoint(eager=True) # checkpoint for ease of use later """

split_rdd_patents = rddPatents.map(lambda row: row.split(',')).filter(lambda row: row[5] != '""') # split with comma separated still has the header , also filter out nan states 
split_rdd_citations = rddCitations.map(lambda row: row.split(','))

rdd_patents_keyed_by_patent = split_rdd_patents.map(lambda row: (row[0], row[5])) # get patent as key 
rdd_citations_keyed_by_citing = split_rdd_citations.map(lambda row: (row[0], row[1])) # get citing as key 
# rdd_citations_keyed_by_cited = split_rdd_citations.map(lambda row: (row[1], row[0])) # get cited as key 

patents_joined_by_citing = rdd_patents_keyed_by_patent.join(rdd_citations_keyed_by_citing) # keyed by citing 
patents_joined_by_citing.checkpoint()
patents_joined_by_citing.take(10) # forces the checkpoint to write to disk 
"""[('3858398', ('""', '1293693')),
 ('3858398', ('""', '1435144')),
 ('3858398', ('""', '3020965')),
 ('3858398', ('""', '3478524')),
 ('3858398', ('""', '3608317')),
 ('3858879', ('"OH"', '2976041')),
 ('3858879', ('"OH"', '3002308')),
 ('3858879', ('"OH"', '3166316')),
 ('3858879', ('"OH"', '3612027')),
 ('3859275', ('""', '3558606'))]"""

'[(\'3858398\', (\'""\', \'1293693\')),\n (\'3858398\', (\'""\', \'1435144\')),\n (\'3858398\', (\'""\', \'3020965\')),\n (\'3858398\', (\'""\', \'3478524\')),\n (\'3858398\', (\'""\', \'3608317\')),\n (\'3858879\', (\'"OH"\', \'2976041\')),\n (\'3858879\', (\'"OH"\', \'3002308\')),\n (\'3858879\', (\'"OH"\', \'3166316\')),\n (\'3858879\', (\'"OH"\', \'3612027\')),\n (\'3859275\', (\'""\', \'3558606\'))]'

In [19]:
patents_joined_by_citing_keyed_by_postate_cited = patents_joined_by_citing.map(lambda row: (
            (row[1][0], row[1][1]), # postate, cited
            row[0]# citing 
        )
    )

rdd_patents_keyed_by_postate_patent = split_rdd_patents.map(lambda row: ((row[5], row[0]), row)) # get postate and patent as key 

full = patents_joined_by_citing_keyed_by_postate_cited.join(rdd_patents_keyed_by_postate_patent) # keyed by postate and cited with value for citing, then cited row 
full.checkpoint()
full.take(5)

[(('"TX"', '3391856'),
  ('5836508',
   ['3391856',
    '1968',
    '3112',
    '1966',
    '"US"',
    '"TX"',
    '',
    '1',
    '',
    '229',
    '6',
    '68',
    '',
    '14',
    '',
    '0.3673',
    '',
    '26.2143',
    '',
    '',
    '',
    '',
    ''])),
 (('"TX"', '3391856'),
  ('5584429',
   ['3391856',
    '1968',
    '3112',
    '1966',
    '"US"',
    '"TX"',
    '',
    '1',
    '',
    '229',
    '6',
    '68',
    '',
    '14',
    '',
    '0.3673',
    '',
    '26.2143',
    '',
    '',
    '',
    '',
    ''])),
 (('"TX"', '3391856'),
  ('4858822',
   ['3391856',
    '1968',
    '3112',
    '1966',
    '"US"',
    '"TX"',
    '',
    '1',
    '',
    '229',
    '6',
    '68',
    '',
    '14',
    '',
    '0.3673',
    '',
    '26.2143',
    '',
    '',
    '',
    '',
    ''])),
 (('"CA"', '4072387'),
  ('5199896',
   ['4072387',
    '1978',
    '6612',
    '1976',
    '"US"',
    '"CA"',
    '535690',
    '2',
    '24',
    '439',
    '4',
    '41',
    '6

In [20]:
keyed_by_citing = full.map(lambda row: (row[1][0], 1))# get the citing as the key again , but the value is the literal 1. We will use this to count 

counts_by_citing = keyed_by_citing.reduceByKey( # reduce by key will execute this operation for each unique key set. The operation is summation of the 1 literals 
        lambda a, b: a + b
    ).sortBy(lambda x: x[1], ascending=False) # sort by value descending 

counts_by_citing.checkpoint()

counts_by_citing.take(10)

[('5959466', 125),
 ('5983822', 103),
 ('6008204', 100),
 ('5952345', 98),
 ('5998655', 96),
 ('5958954', 96),
 ('5936426', 94),
 ('5739256', 90),
 ('5978329', 90),
 ('5913855', 90)]

In [22]:
# now join back to the main 
result = split_rdd_patents.map(lambda row: (row[0], row)).join(counts_by_citing)
result.checkpoint()
result.take(10)

[('3867961',
  (['3867961',
    '1975',
    '5534',
    '1973',
    '"US"',
    '"WI"',
    '84245',
    '2',
    '9',
    '137',
    '6',
    '69',
    '3',
    '3',
    '1',
    '0',
    '0',
    '21.6667',
    '7',
    '0',
    '0',
    '0',
    '0'],
   1)),
 ('3907559',
  (['3907559',
    '1975',
    '5744',
    '1973',
    '"US"',
    '"NY"',
    '635240',
    '2',
    '15',
    '430',
    '1',
    '19',
    '13',
    '1',
    '0.7692',
    '0',
    '0.46',
    '21',
    '8.6154',
    '0.6667',
    '0.4615',
    '0',
    '0'],
   6)),
 ('3922829',
  (['3922829',
    '1975',
    '5814',
    '1973',
    '"US"',
    '"NY"',
    '479225',
    '2',
    '10',
    '52',
    '6',
    '69',
    '6',
    '7',
    '1',
    '0',
    '0.2778',
    '12.4286',
    '5.3333',
    '0',
    '0',
    '0',
    '0'],
   3)),
 ('3972956',
  (['3972956',
    '1976',
    '6059',
    '1975',
    '"US"',
    '"OK"',
    '438920',
    '2',
    '6',
    '585',
    '1',
    '19',
    '6',
    '3',
    '1',
  

In [23]:
flattened_result = result.map(lambda x: [*x[1][0], x[1][1]]) # this is the row, count. The * is a splat operator, and it unpacks the objects . gets rid of the redundant "citing" key 
ordered_results = flattened_result.sortBy(lambda x: x[-1], ascending=False) # order by the last 
csv_rdd = ordered_results.map(lambda row: ",".join(str(item) for item in row)) # convert back to CSV string 
csv_rdd.checkpoint()

csv_rdd.take(10)

['5959466,1999,14515,1997,"US","CA",5310,2,,326,4,46,159,0,1,,0.6186,,4.8868,0.0455,0.044,,,125',
 '5983822,1999,14564,1998,"US","TX",569900,2,,114,5,55,200,0,0.995,,0.7201,,12.45,0,0,,,103',
 '6008204,1999,14606,1998,"US","CA",749584,2,,514,3,31,121,0,1,,0.7415,,5,0.0085,0.0083,,,100',
 '5952345,1999,14501,1997,"US","CA",749584,2,,514,3,31,118,0,1,,0.7442,,5.1102,0,0,,,98',
 '5958954,1999,14515,1997,"US","CA",749584,2,,514,3,31,116,0,1,,0.7397,,5.181,0,0,,,96',
 '5998655,1999,14585,1998,"US","CA",,1,,560,1,14,114,0,1,,0.7387,,5.1667,,,,,96',
 '5936426,1999,14466,1997,"US","CA",5310,2,,326,4,46,178,0,1,,0.58,,11.2303,0.0765,0.073,,,94',
 '5925042,1999,14445,1997,"US","CA",733846,2,,606,3,32,242,0,1,,0.7382,,8.3471,0,0,,,90',
 '5913855,1999,14417,1997,"US","CA",733846,2,,606,3,32,242,0,1,,0.7403,,8.3595,0,0,,,90',
 '5739256,1998,13983,1995,"US","CA",70060,2,15,528,1,15,453,0,1,,0.8232,,15.1104,0.1124,0.1082,,,90']